# Build an Autonomous AI Agent from Scratch (Python, Beginner Friendly)
**David Oluyale | Agentic AI Hub**

In this notebook you build a **research agent**. You give it a goal, like *"compare 3 laptops under $800"*.
It then works **on its own**: it plans, searches, calculates, takes notes, and finally writes a report.

**What you will build, step by step**
1. A single AI call, so you see what a plain chatbot can and cannot do.
2. Four tools: a search over a small laptop catalog, a calculator, and read and write file.
3. The agent loop, from scratch, in about 50 lines of plain Python. No framework.
4. Planning (a to-do list the agent writes for itself) and memory (notes).
5. Guardrails: a step limit, a money budget, and human approval for risky actions.
6. An evaluation: 10 test tasks, scored with real measured results.

Everything runs offline except the AI calls. The "web search" is a small local catalog of **made-up laptops**, so your results are repeatable.

## Part 1. Setup
First we install the one library we need: `anthropic`. It lets Python talk to the AI model.
The `-q` flag just keeps the install quiet.

In [1]:
%pip install -q anthropic

Note: you may need to restart the kernel to use updated packages.


Next we load your API key. **Never paste a key into a notebook.**
On Colab, open the key icon on the left (Secrets), add a secret named `ANTHROPIC_API_KEY`, and switch on notebook access.
This cell reads it from there. On your own computer, it reads it from your environment instead.
We only print **whether** a key was found. We never print the key itself.

In [2]:
import os
try:
    from google.colab import userdata          # only exists on Colab
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass                                        # not on Colab: key must already be in your environment

print("API key found:", bool(os.environ.get("ANTHROPIC_API_KEY")))

API key found: True


Now we create the **client**. Think of it as a phone line to the model.
We pick a small, fast, cheap model, because an agent makes many calls.
We also write down the price per million tokens, so we can measure what every run really costs.

In [3]:
import anthropic, json, time, re

client = anthropic.Anthropic()
MODEL = "claude-haiku-5-5"

# USD per 1 million tokens (input, output) for this model, from the public pricing page.
PRICE_IN, PRICE_OUT = 0.10, 0.50

def cost_usd(usage):
    """Turn the token counts the API reports into dollars."""
    return (usage.input_tokens * PRICE_IN + usage.output_tokens * PRICE_OUT) / 1_000_000

print("Model:", MODEL)

Model: claude-haiku-5-5


## Part 2. What makes an agent autonomous?
Let's start with a plain, single AI call. We ask a question and get one answer back.
This is what a normal chatbot does: **one question in, one answer out**.

In [4]:
def ask(prompt):
    reply = client.messages.create(
        model=MODEL, max_tokens=400, thinking={"type": "disabled"},
        messages=[{"role": "user", "content": prompt}],
    )
    print(f"[{reply.usage.input_tokens} tokens in, {reply.usage.output_tokens} out, ${cost_usd(reply.usage):.6f}]")
    return reply.content[0].text

print(ask("In two sentences: what is an AI agent?"))

[20 tokens in, 97 out, $0.000051]
An AI agent is a system that uses an AI model to pursue goals by taking actions, such as calling tools, browsing, writing and running code, or interacting with other software, rather than just producing a single response. It typically operates in a loop, observing the results of its actions and deciding what to do next until the task is complete.


Now ask it something it **cannot know**: the prices in *our* laptop catalog.
The model has never seen our catalog. A plain chatbot can only guess, or say it does not know.
This is the gap an agent fills: it can **go and look things up**.

In [5]:
print(ask("In our store catalog, which laptop under $800 has the longest battery life? Answer in one sentence."))

[38 tokens in, 65 out, $0.000036]
I don't have access to your store's catalog, so I can't tell you which laptop under $800 has the longest battery life. If you share the catalog data or the product specs, I can compare the battery life figures for you.


### The agent loop
An autonomous agent repeats five steps until the goal is done:

1. **Goal**: you say what you want.
2. **Plan / think**: the model decides the next action.
3. **Act**: it calls a tool, for example a search.
4. **Observe**: it reads the tool's result.
5. **Repeat**: it goes back to step 2, until it can give a final answer.

The model does the thinking. **Our Python code** runs the loop and the tools. That is all an agent is.

## Part 3. Tools
### The data: our "web"
Our agent will search a small catalog of 24 **made-up** laptops. Each one has a price, memory (RAM), storage, battery life, weight and a rating.
This cell loads the catalog file. On Colab it downloads it from the course's GitHub repo the first time.

In [6]:
import urllib.request, pathlib

DATA = pathlib.Path("data/laptops.json")
if not DATA.exists():
    DATA.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/davolu/autonomous-agent-from-scratch/main/data/laptops.json", DATA)

CATALOG = json.loads(DATA.read_text())
print(len(CATALOG), "laptops loaded")
print(json.dumps(CATALOG[0], indent=1))

24 laptops loaded
{
 "id": "L01",
 "name": "Aero 14",
 "brand": "Nimbus",
 "price_usd": 649,
 "ram_gb": 16,
 "storage_gb": 512,
 "cpu": "Ryzen 5",
 "battery_hours": 11,
 "weight_kg": 1.35,
 "screen_in": 14.0,
 "rating": 4.4,
 "tags": [
  "student",
  "office",
  "light"
 ]
}


Let's look at the whole catalog as a small table, so you know what the agent will be searching.
Prices are in US dollars. This is practice data, not real products.

In [7]:
print(f"{'id':4} {'name':16} {'brand':7} {'price':>6} {'RAM':>4} {'SSD':>5} {'batt':>5} {'kg':>5} {'rate':>4}")
for p in CATALOG:
    print(f"{p['id']:4} {p['name']:16} {p['brand']:7} {p['price_usd']:>6} {p['ram_gb']:>4} "
          f"{p['storage_gb']:>5} {p['battery_hours']:>4}h {p['weight_kg']:>5} {p['rating']:>4}")

id   name             brand    price  RAM   SSD  batt    kg rate
L01  Aero 14          Nimbus     649   16   512   11h  1.35  4.4
L02  Aero 14 Pro      Nimbus     899   16  1024   12h  1.38  4.6
L03  Titan G15        Forge     1199   16  1024    5h   2.4  4.5
L04  Titan G17        Forge     1499   32  1024    4h   2.9  4.7
L05  Vector 15        Forge      799    8   512    6h   2.1  4.0
L06  Slate 13         Pebble     549    8   256   10h   1.2  4.1
L07  Slate 14         Pebble     729   16   512   13h   1.3  4.5
L08  Slate 16         Pebble     949   16  1024    9h   1.8  4.3
L09  Bolt 15          Zephyr     599    8   512    7h   1.9  3.9
L10  Bolt 15 Plus     Zephyr     699   16   512    8h   1.9  4.2
L11  Bolt 17          Zephyr     849   16  1024    7h   2.5  4.0
L12  Feather 13       Nimbus     999   16   512   15h  0.99  4.8
L13  Studio 16        Lumen     1399   32  2048   10h  1.95  4.7
L14  Studio 14        Lumen     1099   16  1024   12h  1.45  4.6
L15  Chromo 11        Peb

### Tool 1: search
A tool is just a **normal Python function**. This one is our fake web search.
It looks for words from the query in each laptop's name, brand, chip and tags.
It can also filter by a maximum price, minimum RAM, and brand. Results come back sorted from cheapest to most expensive.

In [8]:
STOP_WORDS = {"laptop", "laptops", "a", "the", "for", "with", "and", "best", "cheap", "under"}

def search_products(query="", max_price=None, min_ram_gb=None, brand=None):
    words = [w for w in re.findall(r"[a-z0-9]+", query.lower()) if w not in STOP_WORDS]
    hits = []
    for p in CATALOG:
        text = " ".join([p["name"], p["brand"], p["cpu"], " ".join(p["tags"])]).lower()
        if words and not any(w in text for w in words):
            continue
        if max_price is not None and p["price_usd"] > max_price:
            continue
        if min_ram_gb is not None and p["ram_gb"] < min_ram_gb:
            continue
        if brand and p["brand"].lower() != brand.lower():
            continue
        hits.append(p)
    hits.sort(key=lambda p: p["price_usd"])
    return {"count": len(hits), "results": hits}

Let's test the search ourselves before we give it to the agent. First: every **gaming** laptop.

In [9]:
r = search_products("gaming")
print(r["count"], "results")
for p in r["results"]:
    print(p["id"], p["name"], "$", p["price_usd"], "| rating", p["rating"])

5 results
L05 Vector 15 $ 799 | rating 4.0
L17 Raptor 16 $ 999 | rating 4.4
L18 Raptor 14 $ 1149 | rating 4.6
L03 Titan G15 $ 1199 | rating 4.5
L04 Titan G17 $ 1499 | rating 4.7


Second test: **student** laptops that cost at most $700 and have at least 16 GB of RAM.
Notice how the filters narrow the list down.

In [10]:
r = search_products("student", max_price=700, min_ram_gb=16)
print(r["count"], "results")
for p in r["results"]:
    print(p["id"], p["name"], "$", p["price_usd"], "|", p["ram_gb"], "GB RAM |", p["battery_hours"], "h battery")

1 results
L01 Aero 14 $ 649 | 16 GB RAM | 11 h battery


Third test: an empty query with only a price filter returns **everything** under that price.
The agent will use this a lot when it needs to compare all options.

In [11]:
r = search_products(max_price=600)
print(r["count"], "results:", [p["name"] for p in r["results"]])

4 results: ['Chromo 11', 'Chromo 14', 'Slate 13', 'Bolt 15']


### Tool 2: calculator
AI models can make small mistakes with arithmetic. So we give the agent a calculator.
We do **not** use Python's `eval`, because that would run any code. Instead we read the maths expression safely, and allow only numbers and + - * / ( ).

In [12]:
import ast, operator

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}

def calculator(expression):
    def walk(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](walk(node.left), walk(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
            return OPS[type(node.op)](walk(node.operand))
        raise ValueError("only numbers and + - * / ( ) are allowed")
    try:
        return {"result": round(walk(ast.parse(expression, mode="eval").body), 4)}
    except Exception as e:
        return {"error": str(e)}

Let's test it. A normal sum works, an average works, and a dangerous input is **refused** instead of being run.

In [13]:
print(calculator("649 + 729 + 779"))
print(calculator("(649 + 729 + 779) / 3"))
print(calculator("__import__('os').getcwd()"))

{'result': 2157}
{'result': 719.0}
{'error': 'only numbers and + - * / ( ) are allowed'}


### Tools 3 and 4: write and read files
The agent will save its final report as a file. These two tools only work inside a `workspace` folder,
so the agent cannot touch any other files on your computer.

In [14]:
WORKSPACE = pathlib.Path("workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)

def safe_path(path):
    full = (WORKSPACE / path).resolve()
    if WORKSPACE not in full.parents and full != WORKSPACE:
        raise ValueError("path is outside the workspace folder")
    return full

def write_file(path, content):
    try:
        f = safe_path(path)
        f.parent.mkdir(parents=True, exist_ok=True)
        f.write_text(content)
        return {"ok": True, "path": str(f.relative_to(WORKSPACE)), "chars": len(content)}
    except Exception as e:
        return {"error": str(e)}

def read_file(path):
    try:
        return {"content": safe_path(path).read_text()[:4000]}
    except Exception as e:
        return {"error": str(e)}

A quick test: write a file, read it back, and try to escape the workspace folder. The last one is blocked.

In [15]:
print(write_file("hello.md", "# Hello agent"))
print(read_file("hello.md"))
print(write_file("../outside.md", "should not work"))

{'ok': True, 'path': 'hello.md', 'chars': 13}
{'content': '# Hello agent'}
{'error': 'path is outside the workspace folder'}


### Describing the tools to the model
The model cannot see our Python code. So we describe each tool in a small JSON **schema**:
its name, what it does, and which inputs it takes. Good descriptions matter a lot: this is how the model decides which tool to use.

In [16]:
TOOLS = [
    {"name": "search_products",
     "description": "Search the laptop catalog. Words in query match name, brand, chip and tags "
                    "(e.g. gaming, student, office, light, creator, budget). Empty query = all laptops. "
                    "Results are sorted by price, cheapest first, with full specs.",
     "input_schema": {"type": "object", "properties": {
         "query": {"type": "string"},
         "max_price": {"type": "number", "description": "maximum price in USD"},
         "min_ram_gb": {"type": "integer"},
         "brand": {"type": "string"}}}},
    {"name": "calculator",
     "description": "Do exact arithmetic. Input like '(649 + 729) / 2'.",
     "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}},
                      "required": ["expression"]}},
    {"name": "write_file",
     "description": "Save text to a file in the workspace, e.g. reports/laptops.md",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}},
                      "required": ["path", "content"]}},
    {"name": "read_file",
     "description": "Read a text file from the workspace.",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}},
]
print([t["name"] for t in TOOLS])

['search_products', 'calculator', 'write_file', 'read_file']


Finally, a small **dispatcher**. When the model asks for a tool by name, this looks up the matching Python function,
runs it with the model's inputs, and turns the result into text the model can read.

In [17]:
TOOL_FUNCS = {"search_products": search_products, "calculator": calculator,
              "write_file": write_file, "read_file": read_file}

def run_tool(name, args):
    if name not in TOOL_FUNCS:
        return json.dumps({"error": f"unknown tool {name}"})
    try:
        return json.dumps(TOOL_FUNCS[name](**args))
    except Exception as e:
        return json.dumps({"error": str(e)})

print(run_tool("calculator", {"expression": "799 - 649"}))

{"result": 150}


## Part 4. The core loop, from scratch
This is the heart of the course. Read it slowly. It is the five steps from Part 2, in code:

- `messages` is the conversation so far. It starts with the **goal**.
- Each turn of the `for` loop, the model **thinks** and either asks for tools or gives a final answer.
- If it asks for tools, we **act** (run them) and send back the results so it can **observe**.
- Then we **repeat**. A `max_steps` limit stops it from looping forever.

In [18]:
SYSTEM = """You are a careful research agent. You work on your own until the goal is done.
Always use the tools to find facts. Never guess prices or specs.
Use the calculator for any arithmetic. When finished, give a short, clear answer."""

def short(text, n=160):
    text = str(text).replace("\n", " ")
    return text if len(text) <= n else text[:n] + " ..."

def run_agent(goal, tools=TOOLS, system=SYSTEM, max_steps=10, verbose=True):
    messages = [{"role": "user", "content": goal}]                 # 1. GOAL
    trace, spent = [], 0.0
    for step in range(1, max_steps + 1):
        reply = client.messages.create(                             # 2. THINK / PLAN
            model=MODEL, max_tokens=2000, system=system, tools=tools,
            messages=messages, thinking={"type": "disabled"})
        spent += cost_usd(reply.usage)
        messages.append({"role": "assistant", "content": reply.content})

        if reply.stop_reason != "tool_use":                         # no tool asked: done
            answer = "".join(b.text for b in reply.content if b.type == "text")
            return {"answer": answer, "steps": step, "trace": trace, "usd": spent, "stopped": "done"}

        results = []
        for block in reply.content:
            if block.type == "tool_use":
                output = run_tool(block.name, block.input)          # 3. ACT
                trace.append((step, block.name, block.input))
                if verbose:
                    print(f"step {step}: {block.name}({short(json.dumps(block.input), 90)})")
                    print(f"        -> {short(output)}")
                results.append({"type": "tool_result",              # 4. OBSERVE
                                "tool_use_id": block.id, "content": output})
        messages.append({"role": "user", "content": results})       # 5. REPEAT
    return {"answer": None, "steps": max_steps, "trace": trace, "usd": spent, "stopped": "max_steps"}

### First run
Let's give the agent its first goal. Watch the printed steps: each line is one tool call the agent decided to make **by itself**.
We never told it which tools to use, or in what order.

In [19]:
t0 = time.time()
result = run_agent("Compare 3 laptops under $800 that are good for students. Which one would you pick and why?")
print("\n=== ANSWER ===")
print(result["answer"])
print(f"\nsteps: {result['steps']} | cost: ${result['usd']:.5f} | time: {time.time() - t0:.1f}s")

step 1: search_products({"query": "student", "max_price": 800})
        -> {"count": 8, "results": [{"id": "L15", "name": "Chromo 11", "brand": "Pebble", "price_usd": 279, "ram_gb": 4, "storage_gb": 64, "cpu": "Celeron", "battery_hours ...


step 2: calculator({"expression": "649 + 689 + 729"})
        -> {"result": 2067}



=== ANSWER ===
I compared three student laptops under $800 from the catalog:

| | Nimbus Aero 14 | Lumen Edge 13 | Pebble Slate 14 |
|---|---|---|---|
| Price | $649 | $689 | $729 |
| CPU | Ryzen 5 | Core i5 | Core i5 |
| RAM / Storage | 16 GB / 512 GB | 8 GB / 512 GB | 16 GB / 512 GB |
| Battery | 11 hrs | 14 hrs | 13 hrs |
| Weight | 1.35 kg | 1.15 kg | 1.3 kg |
| Screen | 14.0" | 13.3" | 14.0" |
| Rating | 4.4 | 4.4 | 4.5 |

**My pick: the Pebble Slate 14 ($729).** It has the highest rating (4.5), 16 GB of RAM, 512 GB of storage, a Core i5, a 13-hour battery, and a 14" screen at a moderate 1.3 kg. It's the most well-rounded for coursework, multitasking, and carrying to class. Its main drawback is that it's the most expensive of the three, at $729.

**Runner-up: the Nimbus Aero 14 ($649).** It's the cheapest of the three and has the same 16 GB / 512 GB configuration, so it's the best choice if you want to save $80. Its battery life (11 hrs) is shorter, though.

**Honorable mention: 

### Reading the trace
The `trace` is a record of every action the agent took. This is how you **debug** an agent:
you look at what it did, step by step, not only at its final answer.

In [20]:
for step, name, args in result["trace"]:
    print(f"step {step} -> {name}: {args}")

step 1 -> search_products: {'query': 'student', 'max_price': 800}
step 2 -> calculator: {'expression': '649 + 689 + 729'}


### A second goal that needs maths
This goal needs the calculator. The agent has to find prices, then work out a difference.
Notice it uses **more than one tool** to finish.

In [21]:
result = run_agent("What is the price difference between the cheapest and the most expensive Forge laptop? "
                   "Use the calculator for the difference.")
print("\n=== ANSWER ===")
print(result["answer"])
print(f"\nsteps: {result['steps']} | cost: ${result['usd']:.5f}")

step 1: search_products({"query": "Forge"})
        -> {"count": 5, "results": [{"id": "L05", "name": "Vector 15", "brand": "Forge", "price_usd": 799, "ram_gb": 8, "storage_gb": 512, "cpu": "Core i5", "battery_hours ...


step 2: calculator({"expression": "1499 - 799"})
        -> {"result": 700}



=== ANSWER ===
The price difference is **$700**. The cheapest Forge laptop is the Vector 15 at $799, and the most expensive is the Titan G17 at $1,499.

steps: 3 | cost: $0.00053


## Part 5. Planning and memory
On bigger goals, agents can lose track. Two simple ideas help a lot:

- **Planning**: the agent writes its own to-do list first, and ticks items off as it goes.
- **Memory**: the agent saves short notes about important facts, so it does not forget them later.

Both are just more tools. They save into a Python dictionary called `MEMORY`.

In [22]:
MEMORY = {"plan": [], "notes": []}

def update_plan(items):
    """Replace the to-do list. items = [{"task": "...", "done": false}, ...]"""
    MEMORY["plan"] = items
    return {"plan": [("[x] " if i.get("done") else "[ ] ") + i["task"] for i in items]}

def save_note(note):
    MEMORY["notes"].append(note)
    return {"saved": True, "notes_so_far": len(MEMORY["notes"])}

def read_notes():
    return {"notes": MEMORY["notes"]}

TOOL_FUNCS.update({"update_plan": update_plan, "save_note": save_note, "read_notes": read_notes})
print(update_plan([{"task": "test the plan tool", "done": True}]))
MEMORY = {"plan": [], "notes": []}

{'plan': ['[x] test the plan tool']}


We describe the three new tools to the model, just like before, and add them to the tool list.

In [23]:
MEMORY_TOOLS = [
    {"name": "update_plan",
     "description": "Write or update your to-do list. Call it first with all steps, then again as you finish them.",
     "input_schema": {"type": "object", "properties": {"items": {"type": "array", "items": {
         "type": "object", "properties": {"task": {"type": "string"}, "done": {"type": "boolean"}},
         "required": ["task", "done"]}}}, "required": ["items"]}},
    {"name": "save_note",
     "description": "Save one short fact you found (e.g. 'L07 Slate 14: $729, 13h battery').",
     "input_schema": {"type": "object", "properties": {"note": {"type": "string"}}, "required": ["note"]}},
    {"name": "read_notes",
     "description": "Read all notes you saved so far.",
     "input_schema": {"type": "object", "properties": {}}},
]
PLANNER_TOOLS = TOOLS + MEMORY_TOOLS
print([t["name"] for t in PLANNER_TOOLS])

['search_products', 'calculator', 'write_file', 'read_file', 'update_plan', 'save_note', 'read_notes']


The system prompt now tells the agent **how to work**: plan first, take notes, read the notes before writing,
and save the final report as a file. The loop itself does not change at all.

In [24]:
PLANNER_SYSTEM = SYSTEM + """
How to work:
1. First call update_plan with your full to-do list.
2. Save each important fact with save_note.
3. Mark plan items done with update_plan as you finish them.
4. Before writing, call read_notes. Then save a markdown report with write_file in reports/.
5. Finish with a short summary."""
print(PLANNER_SYSTEM)

You are a careful research agent. You work on your own until the goal is done.
Always use the tools to find facts. Never guess prices or specs.
Use the calculator for any arithmetic. When finished, give a short, clear answer.
How to work:
1. First call update_plan with your full to-do list.
2. Save each important fact with save_note.
3. Mark plan items done with update_plan as you finish them.
4. Before writing, call read_notes. Then save a markdown report with write_file in reports/.
5. Finish with a short summary.


### Running the planning agent
Now a bigger goal: research, compare, and write a report. Watch for the plan at the start, the notes in the middle, and the file at the end.

In [25]:
MEMORY = {"plan": [], "notes": []}
t0 = time.time()
result = run_agent("Research the 3 best laptops under $800 for a university student who travels a lot "
                   "(battery life and weight matter). Write a comparison report to reports/student_laptops.md.",
                   tools=PLANNER_TOOLS, system=PLANNER_SYSTEM, max_steps=20)
print("\n=== ANSWER ===")
print(result["answer"])
print(f"\nsteps: {result['steps']} | cost: ${result['usd']:.5f} | time: {time.time() - t0:.1f}s")

step 1: update_plan({"items": [{"task": "Search laptop catalog for options under $800 (student, travel, batter ...)
        -> {"plan": ["[ ] Search laptop catalog for options under $800 (student, travel, battery, light)", "[ ] Save key facts (price, battery, weight) as notes for candid ...
step 1: search_products({"query": "student light battery", "max_price": 800})
        -> {"count": 8, "results": [{"id": "L15", "name": "Chromo 11", "brand": "Pebble", "price_usd": 279, "ram_gb": 4, "storage_gb": 64, "cpu": "Celeron", "battery_hours ...
step 1: search_products({"query": "travel", "max_price": 800})
        -> {"count": 0, "results": []}


step 2: save_note({"note": "L20 Edge 13 (Lumen): $689, 14h battery, 1.15 kg, 13.3in, Core i5, 8GB/512GB, rat ...)
        -> {"saved": true, "notes_so_far": 1}
step 2: save_note({"note": "L07 Slate 14 (Pebble): $729, 13h battery, 1.3 kg, 14in, Core i5, 16GB/512GB, rat ...)
        -> {"saved": true, "notes_so_far": 2}
step 2: save_note({"note": "L01 Aero 14 (Nimbus): $649, 11h battery, 1.35 kg, 14in, Ryzen 5, 16GB/512GB, rat ...)
        -> {"saved": true, "notes_so_far": 3}
step 2: save_note({"note": "L06 Slate 13 (Pebble): $549, 10h battery, 1.2 kg, 13.3in, Core i3, 8GB/256GB, ra ...)
        -> {"saved": true, "notes_so_far": 4}
step 2: save_note({"note": "Excluded: L19 Edge 15 ($779, 9h, 1.7 kg) worst on battery and weight; L24 Nova 1 ...)
        -> {"saved": true, "notes_so_far": 5}


step 3: calculator({"expression": "(14 + 13 + 11) / 3"})
        -> {"result": 12.6667}
step 3: calculator({"expression": "(1.15 + 1.3 + 1.35) / 3"})
        -> {"result": 1.2667}
step 3: read_notes({})
        -> {"notes": ["L20 Edge 13 (Lumen): $689, 14h battery, 1.15 kg, 13.3in, Core i5, 8GB/512GB, rating 4.4", "L07 Slate 14 (Pebble): $729, 13h battery, 1.3 kg, 14in, C ...


step 4: write_file({"path": "reports/student_laptops.md", "content": "# Top 3 Laptops Under $800 for a Travel ...)
        -> {"ok": true, "path": "reports/student_laptops.md", "chars": 2252}
step 4: update_plan({"items": [{"task": "Search laptop catalog for options under $800", "done": true}, {"task" ...)
        -> {"plan": ["[x] Search laptop catalog for options under $800", "[x] Save key facts as notes", "[x] Select top 3 laptops", "[x] Read notes and compute comparisons ...



=== ANSWER ===
The report is saved to `reports/student_laptops.md`. The top three laptops under $800 for a traveling student:

1. **Lumen Edge 13, $689**: 14 h battery, 1.15 kg, 13.3" screen, Core i5, 8 GB RAM. Best for travel.
2. **Pebble Slate 14, $729**: 13 h battery, 1.30 kg, 14" screen, Core i5, 16 GB RAM, rating 4.5. Best all-round, $40 more than the Edge 13.
3. **Nimbus Aero 14, $649**: 11 h battery, 1.35 kg, 14" screen, Ryzen 5, 16 GB RAM. Best value, but the shortest battery and heaviest of the three.

I recommend the Edge 13 for heavy travel. If you need more RAM and screen space, get the Slate 14.

steps: 5 | cost: $0.00337 | time: 11.7s


Let's look inside the agent's memory. First its **plan**, the to-do list it wrote for itself, as it was at the end.

In [26]:
for item in MEMORY["plan"]:
    print(("[x] " if item.get("done") else "[ ] ") + item["task"])

[x] Search laptop catalog for options under $800
[x] Save key facts as notes
[x] Select top 3 laptops
[x] Read notes and compute comparisons
[x] Write comparison report to reports/student_laptops.md
[x] Give short summary to user


And here are the **notes** it saved while it was working.

In [27]:
for i, note in enumerate(MEMORY["notes"], 1):
    print(f"{i}. {note}")

1. L20 Edge 13 (Lumen): $689, 14h battery, 1.15 kg, 13.3in, Core i5, 8GB/512GB, rating 4.4
2. L07 Slate 14 (Pebble): $729, 13h battery, 1.3 kg, 14in, Core i5, 16GB/512GB, rating 4.5
3. L01 Aero 14 (Nimbus): $649, 11h battery, 1.35 kg, 14in, Ryzen 5, 16GB/512GB, rating 4.4
4. L06 Slate 13 (Pebble): $549, 10h battery, 1.2 kg, 13.3in, Core i3, 8GB/256GB, rating 4.1
5. Excluded: L19 Edge 15 ($779, 9h, 1.7 kg) worst on battery and weight; L24 Nova 14 ($619, 10h, 1.4 kg); L16 Chromo 14 ($379, 11h, 1.4 kg); L15 Chromo 11 ($279, 12h, 1.1 kg, 4GB RAM, Celeron, rating 3.6)


Finally, the **report** it wrote. This file was created by the agent, not by us.

In [28]:
print((WORKSPACE / "reports/student_laptops.md").read_text())

# Top 3 Laptops Under $800 for a Traveling University Student

Priorities: battery life and weight, with enough performance for coursework.

## Recommendations

| Rank | Laptop | Price | Battery | Weight | Screen | CPU | RAM / Storage | Rating |
|------|--------|-------|---------|--------|--------|-----|---------------|--------|
| 1 | Lumen Edge 13 (L20) | $689 | 14 h | 1.15 kg | 13.3 in | Core i5 | 8 GB / 512 GB | 4.4 |
| 2 | Pebble Slate 14 (L07) | $729 | 13 h | 1.30 kg | 14.0 in | Core i5 | 16 GB / 512 GB | 4.5 |
| 3 | Nimbus Aero 14 (L01) | $649 | 11 h | 1.35 kg | 14.0 in | Ryzen 5 | 16 GB / 512 GB | 4.4 |

## Summary by Pick

**1. Lumen Edge 13: best for travel.** It has the longest battery (14 h) and the lightest weight (1.15 kg) of the group, and the smallest screen. At $689 it is also the second cheapest of the three. Only 8 GB RAM, which is adequate for most coursework.

**2. Pebble Slate 14: best all-round.** 13 h battery, 16 GB RAM, the highest rating (4.5), and a 14-inch sc

## Part 6. Guardrails
An agent that acts on its own needs limits. We add three:

1. **Max steps**: stop after N steps, so it can never loop forever.
2. **Budget cap**: stop when the money spent reaches a limit.
3. **Human approval**: risky tools (here, writing files) need a "yes" first.

First, the approval rule. In `policy` mode it approves writes only to markdown files in `reports/`.
In `ask` mode it asks you, the human, by typing y or n. Every decision is printed, so you can audit it.

In [29]:
RISKY_TOOLS = {"write_file"}
APPROVAL_MODE = "policy"        # change to "ask" to approve each risky action yourself

def approve(name, args):
    if name not in RISKY_TOOLS:
        return True
    if APPROVAL_MODE == "ask":
        return input(f"Allow {name} {args.get('path')}? (y/n) ").strip().lower() == "y"
    path = str(args.get("path", ""))
    return path.startswith("reports/") and path.endswith(".md") and ".." not in path

print(approve("calculator", {"expression": "1+1"}))
print(approve("write_file", {"path": "reports/ok.md"}))
print(approve("write_file", {"path": "notes/secret.txt"}))

True
True
False


Now the **safe agent**. It is the same loop as before, with three additions, marked with comments:
a budget check before each model call, an approval check before each risky tool, and a clear reason when it stops.

In [30]:
def run_agent_safe(goal, tools=PLANNER_TOOLS, system=PLANNER_SYSTEM, max_steps=15, budget_usd=0.02, verbose=True):
    messages = [{"role": "user", "content": goal}]
    trace, spent = [], 0.0
    for step in range(1, max_steps + 1):
        if spent >= budget_usd:                                          # GUARDRAIL 2: budget
            return {"answer": None, "steps": step - 1, "trace": trace, "usd": spent, "stopped": "budget"}
        reply = client.messages.create(model=MODEL, max_tokens=2000, system=system, tools=tools,
                                       messages=messages, thinking={"type": "disabled"})
        spent += cost_usd(reply.usage)
        messages.append({"role": "assistant", "content": reply.content})
        if reply.stop_reason != "tool_use":
            answer = "".join(b.text for b in reply.content if b.type == "text")
            return {"answer": answer, "steps": step, "trace": trace, "usd": spent, "stopped": "done"}
        results = []
        for block in reply.content:
            if block.type != "tool_use":
                continue
            if approve(block.name, block.input):                         # GUARDRAIL 3: approval
                output = run_tool(block.name, block.input)
            else:
                output = json.dumps({"error": "DENIED by human approval policy. "
                                              "Only markdown files in reports/ may be written."})
            trace.append((step, block.name, block.input))
            if verbose:
                print(f"step {step}: {block.name}({short(json.dumps(block.input), 90)})")
                print(f"        -> {short(output)}")
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})
        messages.append({"role": "user", "content": results})
    return {"answer": None, "steps": max_steps, "trace": trace, "usd": spent, "stopped": "max_steps"}  # GUARDRAIL 1

### Guardrail test 1: max steps
We give a big goal but allow only **2 steps**. The agent must stop early, and it tells us why.

In [31]:
MEMORY = {"plan": [], "notes": []}
r = run_agent_safe("Write a full report on every laptop brand in the catalog.", max_steps=2)
print("\nstopped because:", r["stopped"], "| steps:", r["steps"], f"| cost: ${r['usd']:.5f}")

step 1: update_plan({"items": [{"task": "Search full laptop catalog to identify all brands and their specs", " ...)
        -> {"plan": ["[ ] Search full laptop catalog to identify all brands and their specs", "[ ] Save key facts per brand as notes", "[ ] Read notes and compile per-bran ...
step 1: search_products({"query": ""})
        -> {"count": 24, "results": [{"id": "L15", "name": "Chromo 11", "brand": "Pebble", "price_usd": 279, "ram_gb": 4, "storage_gb": 64, "cpu": "Celeron", "battery_hour ...


step 2: save_note({"note": "Pebble (4 models): Chromo 11 $279 (3.6), Chromo 14 $379 (3.8), Slate 13 $549 (4. ...)
        -> {"saved": true, "notes_so_far": 1}
step 2: save_note({"note": "Zephyr (4 models): Bolt 15 $599 (3.9, 7h), Bolt 15 Plus $699 (4.2, 8h), Workhors ...)
        -> {"saved": true, "notes_so_far": 2}
step 2: save_note({"note": "Nimbus (5 models): Nova 14 $619 (4.0), Aero 14 $649 (4.4, 16GB, 11h), Aero 14 Pr ...)
        -> {"saved": true, "notes_so_far": 3}
step 2: save_note({"note": "Lumen (4 models): Edge 13 $689 (4.4, 14h, 1.15kg), Edge 15 $779 (4.3, 9h), Studi ...)
        -> {"saved": true, "notes_so_far": 4}
step 2: save_note({"note": "Forge (4 models): Vector 15 $799 (4.0, 6h, gaming budget), Raptor 16 $999 (4.4,  ...)
        -> {"saved": true, "notes_so_far": 5}

stopped because: max_steps | steps: 2 | cost: $0.00118


### Guardrail test 2: budget cap
Now we set a tiny budget: one tenth of a cent. The agent stops as soon as it has spent that much.

In [32]:
MEMORY = {"plan": [], "notes": []}
r = run_agent_safe("Write a full report on every laptop brand in the catalog.", budget_usd=0.001)
print("\nstopped because:", r["stopped"], "| steps:", r["steps"], f"| cost: ${r['usd']:.5f}")

step 1: update_plan({"items": [{"task": "Search full laptop catalog to identify all brands and their specs", " ...)
        -> {"plan": ["[ ] Search full laptop catalog to identify all brands and their specs", "[ ] Save key facts per brand/model as notes", "[ ] Read notes and compile pe ...
step 1: search_products({"query": ""})
        -> {"count": 24, "results": [{"id": "L15", "name": "Chromo 11", "brand": "Pebble", "price_usd": 279, "ram_gb": 4, "storage_gb": 64, "cpu": "Celeron", "battery_hour ...


step 2: save_note({"note": "Pebble (5 models: Chromo 11 $279 3.6\u2605; Chromo 14 $379 3.8\u2605; Slate 13 $ ...)
        -> {"saved": true, "notes_so_far": 1}
step 2: save_note({"note": "Zephyr (4 models: Bolt 15 $599 3.9\u2605 7h; Bolt 15 Plus $699 4.2\u2605 8h; Wor ...)
        -> {"saved": true, "notes_so_far": 2}
step 2: save_note({"note": "Nimbus (5 models: Nova 14 $619 4.0\u2605 10h; Aero 14 $649 4.4\u2605 11h 1.35kg; ...)
        -> {"saved": true, "notes_so_far": 3}
step 2: save_note({"note": "Lumen (4 models: Edge 13 $689 4.4\u2605 14h 1.15kg; Edge 15 $779 4.3\u2605 9h; S ...)
        -> {"saved": true, "notes_so_far": 4}
step 2: save_note({"note": "Forge (5 models: Vector 15 $799 4.0\u2605 6h; Raptor 16 $999 4.4\u2605 6h; Rapto ...)
        -> {"saved": true, "notes_so_far": 5}

stopped because: budget | steps: 2 | cost: $0.00121


### Guardrail test 3: human approval
We ask the agent to save a file in a place that is **not allowed**. The approval rule denies it,
and the agent receives that "denied" message as an observation, so it can adapt.

In [33]:
MEMORY = {"plan": [], "notes": []}
r = run_agent_safe("Find the cheapest gaming laptop and save a one-line summary to notes/gaming.txt.",
                   max_steps=10)
print("\n=== ANSWER ===")
print(r["answer"])
print("\nstopped because:", r["stopped"], f"| cost: ${r['usd']:.5f}")
print("files in workspace:", sorted(str(p.relative_to(WORKSPACE)) for p in WORKSPACE.rglob("*") if p.is_file()))

step 1: update_plan({"items": [{"task": "Search the catalog for gaming laptops", "done": false}, {"task": "Ide ...)
        -> {"plan": ["[ ] Search the catalog for gaming laptops", "[ ] Identify the cheapest gaming laptop and save key facts as notes", "[ ] Review notes before writing", ...
step 1: search_products({"query": "gaming"})
        -> {"count": 5, "results": [{"id": "L05", "name": "Vector 15", "brand": "Forge", "price_usd": 799, "ram_gb": 8, "storage_gb": 512, "cpu": "Core i5", "battery_hours ...


step 2: save_note({"note": "Cheapest gaming laptop: Forge Vector 15 (L05), $799, Core i5, 8GB RAM, 512GB sto ...)
        -> {"saved": true, "notes_so_far": 1}
step 2: update_plan({"items": [{"task": "Search the catalog for gaming laptops", "done": true}, {"task": "Iden ...)
        -> {"plan": ["[x] Search the catalog for gaming laptops", "[x] Identify the cheapest gaming laptop and save key facts as notes", "[ ] Review notes before writing", ...
step 2: read_notes({})
        -> {"notes": ["Cheapest gaming laptop: Forge Vector 15 (L05), $799, Core i5, 8GB RAM, 512GB storage, 6h battery, 4.0 rating"]}


step 3: write_file({"path": "notes/gaming.txt", "content": "Cheapest gaming laptop: Forge Vector 15 (L05) at  ...)
        -> {"error": "DENIED by human approval policy. Only markdown files in reports/ may be written."}
step 3: update_plan({"items": [{"task": "Search the catalog for gaming laptops", "done": true}, {"task": "Iden ...)
        -> {"plan": ["[x] Search the catalog for gaming laptops", "[x] Identify the cheapest gaming laptop and save key facts as notes", "[x] Review notes before writing", ...



=== ANSWER ===
The cheapest gaming laptop is the **Forge Vector 15 (L05) at $799**, with a Core i5, 8GB RAM, 512GB storage, 6-hour battery, 15.6" screen, and a 4.0 rating.

I could not save the summary to `notes/gaming.txt`. The write was denied by the human approval policy, which only allows markdown files in `reports/`. The one-line summary I intended to save is:

> Cheapest gaming laptop: Forge Vector 15 (L05) at $799 - Core i5, 8GB RAM, 512GB storage, 6h battery, 15.6in, rating 4.0.

If you want the file saved, either approve the write to `notes/gaming.txt` or let me save it as a markdown report at `reports/gaming.md` instead. I also marked the last plan step as done even though that write failed, so that step is not actually complete.

stopped because: done | cost: $0.00152
files in workspace: ['hello.md', 'reports/student_laptops.md']


### Worked example: a full guarded run
Let's use the finished, safe agent on a realistic request. It plans, searches, takes notes, does maths,
and writes a report, all inside its limits.

In [34]:
MEMORY = {"plan": [], "notes": []}
t0 = time.time()
r = run_agent_safe("I edit videos and have $1,200. Find the 3 best laptops for content creators within budget. "
                   "Include the average price of the three (use the calculator). Save the report to reports/creator.md.",
                   max_steps=20, budget_usd=0.05)
print("\n=== ANSWER ===")
print(r["answer"])
print(f"\nstopped: {r['stopped']} | steps: {r['steps']} | cost: ${r['usd']:.5f} | time: {time.time() - t0:.1f}s")

step 1: update_plan({"items": [{"task": "Search laptop catalog for creator/video editing laptops under $1200", ...)
        -> {"plan": ["[ ] Search laptop catalog for creator/video editing laptops under $1200", "[ ] Save key facts from results as notes", "[ ] Select top 3 laptops for v ...
step 1: search_products({"query": "creator", "max_price": 1200})
        -> {"count": 4, "results": [{"id": "L23", "name": "Nova 16", "brand": "Nimbus", "price_usd": 759, "ram_gb": 16, "storage_gb": 1024, "cpu": "Ryzen 7", "battery_hour ...
step 1: search_products({"query": "video editing", "max_price": 1200})
        -> {"count": 0, "results": []}


step 2: save_note({"note": "L23 Nova 16 (Nimbus): $759, Ryzen 7, 16GB RAM, 1TB, 8h battery, 1.85kg, 16in, ra ...)
        -> {"saved": true, "notes_so_far": 1}
step 2: save_note({"note": "L02 Aero 14 Pro (Nimbus): $899, Ryzen 7, 16GB RAM, 1TB, 12h battery, 1.38kg, 14i ...)
        -> {"saved": true, "notes_so_far": 2}
step 2: save_note({"note": "L08 Slate 16 (Pebble): $949, Core i7, 16GB RAM, 1TB, 9h battery, 1.8kg, 16in, ra ...)
        -> {"saved": true, "notes_so_far": 3}
step 2: save_note({"note": "L14 Studio 14 (Lumen): $1099, Core Ultra 7, 16GB RAM, 1TB, 12h battery, 1.45kg,  ...)
        -> {"saved": true, "notes_so_far": 4}
step 2: search_products({"query": "gaming", "max_price": 1200})
        -> {"count": 4, "results": [{"id": "L05", "name": "Vector 15", "brand": "Forge", "price_usd": 799, "ram_gb": 8, "storage_gb": 512, "cpu": "Core i5", "battery_hours ...
step 2: search_products({"query": "", "max_price": 1200})
        -> {"count": 22, "results": [{"id": "L15", "name": "Ch

step 3: save_note({"note": "L18 Raptor 14 (Forge): $1149, Ryzen 9, 16GB RAM, 1TB, 7h battery, 1.7kg, 14in, r ...)
        -> {"saved": true, "notes_so_far": 5}
step 3: save_note({"note": "L12 Feather 13 (Nimbus): $999, Core Ultra 7, 16GB RAM, 512GB, 15h battery, 0.99k ...)
        -> {"saved": true, "notes_so_far": 6}
step 3: calculator({"expression": "(1099 + 899 + 1149) / 3"})
        -> {"result": 1049.0}


step 4: calculator({"expression": "(1099 + 899 + 999) / 3"})
        -> {"result": 999.0}
step 4: read_notes({})
        -> {"notes": ["L23 Nova 16 (Nimbus): $759, Ryzen 7, 16GB RAM, 1TB, 8h battery, 1.85kg, 16in, rating 4.3, tags office/creator/big-screen", "L02 Aero 14 Pro (Nimbus) ...


step 5: update_plan({"items": [{"task": "Search laptop catalog for creator/video editing laptops under $1200", ...)
        -> {"plan": ["[x] Search laptop catalog for creator/video editing laptops under $1200", "[x] Save key facts from results as notes", "[x] Select top 3 laptops for v ...
step 5: write_file({"path": "reports/creator.md", "content": "# Best Laptops for Video Editing Under $1,200\n ...)
        -> {"ok": true, "path": "reports/creator.md", "chars": 1682}



=== ANSWER ===
The report is saved to `reports/creator.md`.

**Top 3 laptops for video editing under $1,200:**
1. **Lumen Studio 14**: $1,099, Core Ultra 7, 16GB RAM, 1TB, 12h battery, 4.6 rating. The only "creator"-tagged option with a strong CPU and 1TB storage at this level.
2. **Nimbus Aero 14 Pro**: $899, Ryzen 7, 16GB RAM, 1TB, 12h battery, 1.38kg, 4.6 rating. Best value.
3. **Nimbus Feather 13**: $999, Core Ultra 7, 16GB RAM, 512GB, 15h battery, 4.8 rating. Highest rated, but the smallest storage.

**Average price of the three:** $999.00

Caveat: the catalog doesn't list GPU or color-accuracy specs, so check those separately if color grading is important to your work.

stopped: done | steps: 6 | cost: $0.00508 | time: 12.6s


Here is the creator report the agent saved.

In [35]:
print((WORKSPACE / "reports/creator.md").read_text())

# Best Laptops for Video Editing Under $1,200

Budget: $1,200. Source: laptop catalog search (all laptops priced at or under $1,200).

## Top 3 Picks

| Rank | Laptop | Price | CPU | RAM | Storage | Screen | Battery | Weight | Rating |
|---|---|---|---|---|---|---|---|---|---|
| 1 | Lumen Studio 14 (L14) | $1,099 | Core Ultra 7 | 16GB | 1TB | 14in | 12h | 1.45kg | 4.6 |
| 2 | Nimbus Aero 14 Pro (L02) | $899 | Ryzen 7 | 16GB | 1TB | 14in | 12h | 1.38kg | 4.6 |
| 3 | Nimbus Feather 13 (L12) | $999 | Core Ultra 7 | 16GB | 512GB | 13.3in | 15h | 0.99kg | 4.8 |

### Why these three
- **Lumen Studio 14**: the only laptop in the catalog explicitly tagged "creator" with a Core Ultra 7 and 1TB storage. Best for sustained editing workloads.
- **Nimbus Aero 14 Pro**: strong Ryzen 7 performance, 1TB storage, and a light 1.38kg chassis, at the lowest price of the three.
- **Nimbus Feather 13**: highest rating (4.8), Core Ultra 7, and the longest battery life (15h). Its 512GB storage is the smallest

## Part 7. Evaluation
How do we know the agent is any good? We **test it**, like any other software.
We write 10 tasks. For each one, we compute the correct answer **ourselves, with plain Python** from the catalog.
Then we compare the agent's answer to it. No opinions, just measured results.

In [36]:
C = CATALOG
def cheapest(rows): return min(rows, key=lambda p: p["price_usd"])

TASKS = [
    ("Which laptop is the cheapest in the whole catalog?", cheapest(C)["id"]),
    ("Which is the cheapest laptop with at least 16 GB of RAM?", cheapest([p for p in C if p["ram_gb"] >= 16])["id"]),
    ("Which laptop under $800 has the longest battery life?",
        max([p for p in C if p["price_usd"] < 800], key=lambda p: p["battery_hours"])["id"]),
    ("Which is the lightest laptop under $1000?",
        min([p for p in C if p["price_usd"] < 1000], key=lambda p: p["weight_kg"])["id"]),
    ("Which gaming laptop has the highest rating?", max([p for p in C if "gaming" in p["tags"]], key=lambda p: p["rating"])["id"]),
    ("Which is the cheapest laptop with at least 1 TB (1024 GB) of storage?",
        cheapest([p for p in C if p["storage_gb"] >= 1024])["id"]),
    ("Which laptop under $600 has the best rating?", max([p for p in C if p["price_usd"] < 600], key=lambda p: p["rating"])["id"]),
    ("What is the total price in USD of the two cheapest laptops?", sum(sorted(p["price_usd"] for p in C)[:2])),
    ("What is the average price in USD of all Forge laptops?",
        round(sum(p["price_usd"] for p in C if p["brand"] == "Forge") / len([p for p in C if p["brand"] == "Forge"]), 2)),
    ("How many laptops cost under $700 and have at least 16 GB of RAM?",
        len([p for p in C if p["price_usd"] < 700 and p["ram_gb"] >= 16])),
]
for i, (q, a) in enumerate(TASKS, 1):
    print(f"{i:2}. {q}  ->  expected: {a}")

 1. Which laptop is the cheapest in the whole catalog?  ->  expected: L15
 2. Which is the cheapest laptop with at least 16 GB of RAM?  ->  expected: L01
 3. Which laptop under $800 has the longest battery life?  ->  expected: L20
 4. Which is the lightest laptop under $1000?  ->  expected: L12
 5. Which gaming laptop has the highest rating?  ->  expected: L04
 6. Which is the cheapest laptop with at least 1 TB (1024 GB) of storage?  ->  expected: L23
 7. Which laptop under $600 has the best rating?  ->  expected: L06
 8. What is the total price in USD of the two cheapest laptops?  ->  expected: 658
 9. What is the average price in USD of all Forge laptops?  ->  expected: 1129.0
10. How many laptops cost under $700 and have at least 16 GB of RAM?  ->  expected: 2


The agent must end its answer with a line like `FINAL ANSWER: L07` or `FINAL ANSWER: 658`.
This **checker** reads that line and compares it with the expected answer. Numbers may differ by at most 0.01.

In [37]:
EVAL_SYSTEM = SYSTEM + "\nEnd your reply with one line: FINAL ANSWER: <laptop id like L07, or a number>"

def check(answer, expected):
    m = re.search(r"FINAL ANSWER:\s*\$?([A-Za-z0-9.,]+)", answer or "")
    if not m:
        return False, None
    got = m.group(1).rstrip(".,").replace(",", "")
    if isinstance(expected, str):
        return got.upper() == expected, got
    try:
        return abs(float(got) - expected) <= 0.01, got
    except ValueError:
        return False, got

print(check("blah\nFINAL ANSWER: L15", "L15"), check("FINAL ANSWER: $1,129.00", 1129.0), check("no idea", 2))

(True, 'L15') (True, '1129.00') (False, None)


Now we run all 10 tasks. For each one we record: pass or fail, the agent's answer, the steps it took, the cost and the time.
This takes a minute or two, because the agent works through each task on its own.

In [38]:
rows = []
for i, (q, expected) in enumerate(TASKS, 1):
    t0 = time.time()
    r = run_agent_safe(q, tools=TOOLS, system=EVAL_SYSTEM, max_steps=8, budget_usd=0.02, verbose=False)
    ok, got = check(r["answer"], expected)
    rows.append({"task": i, "pass": ok, "got": got, "expected": expected, "steps": r["steps"],
                 "usd": round(r["usd"], 5), "sec": round(time.time() - t0, 1), "stopped": r["stopped"]})
    print(f"task {i:2}: {'PASS' if ok else 'FAIL'}  got={got}  expected={expected}  "
          f"steps={r['steps']}  ${r['usd']:.5f}  {rows[-1]['sec']}s")

task  1: PASS  got=L15  expected=L15  steps=2  $0.00060  1.3s


task  2: PASS  got=L01  expected=L01  steps=2  $0.00048  1.3s


task  3: PASS  got=L20  expected=L20  steps=2  $0.00045  1.6s


task  4: PASS  got=L12  expected=L12  steps=2  $0.00053  1.4s


task  5: PASS  got=L04  expected=L04  steps=2  $0.00032  1.1s


task  6: PASS  got=L23  expected=L23  steps=2  $0.00061  1.4s


task  7: PASS  got=L06  expected=L06  steps=2  $0.00034  1.4s


task  8: PASS  got=658  expected=658  steps=3  $0.00103  1.7s


task  9: PASS  got=1129  expected=1129.0  steps=3  $0.00054  1.6s


task 10: PASS  got=2  expected=2  steps=2  $0.00037  1.7s


### The score
Here is the summary, measured from the run above: how many tasks passed, the total cost, and the average steps and time per task.

In [39]:
passed = sum(r["pass"] for r in rows)
print(f"Score: {passed}/{len(rows)} tasks passed")
print(f"Total cost: ${sum(r['usd'] for r in rows):.5f}")
print(f"Average steps per task: {sum(r['steps'] for r in rows) / len(rows):.1f}")
print(f"Average time per task: {sum(r['sec'] for r in rows) / len(rows):.1f}s")
pathlib.Path("results").mkdir(exist_ok=True)
json.dump(rows, open("results/eval_results.json", "w"), indent=1)
failed = [r for r in rows if not r["pass"]]
print("Failed tasks:", [(r["task"], r["got"], r["expected"]) for r in failed] or "none")

Score: 10/10 tasks passed
Total cost: $0.00527
Average steps per task: 2.2
Average time per task: 1.4s
Failed tasks: none


## Part 8. Recap and next steps
You built a real autonomous agent from scratch:

- **The loop**: goal, think, act, observe, repeat, in about 50 lines of Python.
- **Tools**: search, calculator, read and write file, each a plain function plus a short description.
- **Planning and memory**: a to-do list and notes, added as more tools.
- **Guardrails**: max steps, a budget cap, and human approval for risky actions.
- **Evaluation**: 10 tasks with answers we computed ourselves, scored automatically.

**Try next**
- Swap `search_products` for a real web search API.
- Add your own tool, for example sending a draft email that always needs approval.
- Add harder tasks to the evaluation, and re-run it every time you change the prompt.

*David Oluyale | Agentic AI Hub*